# FewGaze geometry result, replicated from scratch

Goal: check the paper's headline number for the geometry method (AUROC ≈ 0.81 with all training labels,
held-out subjects) without using any of the repo's code. Nothing here imports `fewgaze` or reads a repo file.
Steps: parse Columbia Gaze filenames → my own MediaPipe eye/head features → GroupKFold by subject →
logistic regression (linear and degree-2) → AUROC on unseen people.

In [1]:
import hashlib, re, time, urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import sklearn
from mediapipe.tasks.python import BaseOptions, vision
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

DATA = Path(r"../data/raw/Columbia Gaze Data Set")   # folder with 0001 ... 0056
MODEL = Path("face_landmarker.task")
CACHE = Path("replication_features.csv")
if not MODEL.exists():
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", MODEL)
print("mediapipe", mp.__version__, "| sklearn", sklearn.__version__, "| numpy", np.__version__)

mediapipe 0.10.21 | sklearn 1.9.1 | numpy 1.26.4


## 1. Load images and labels from filenames
`0001_2m_-15P_10V_-5H.jpg` = subject, distance, head pose, vertical gaze, horizontal gaze. Positive = gaze (0°, 0°).

In [2]:
pat = re.compile(r"(\d{4})_2m_(-?\d+)P_(-?\d+)V_(-?\d+)H\.jpg$", re.I)
rows = []
for f in sorted(DATA.rglob("*.jpg")):
    m = pat.match(f.name)
    if m:
        assert m[1] == f.parent.name, f
        rows.append(dict(path=str(f), subject=m[1], pose=int(m[2]), v=int(m[3]), h=int(m[4])))
df = pd.DataFrame(rows)
df["y"] = ((df.v == 0) & (df.h == 0)).astype(int)
print(len(df), "images,", df.subject.nunique(), "subjects,", df.y.sum(), "positives")
print(df.groupby("subject").size().describe()[["min", "max"]].to_dict())

5880 images, 56 subjects, 280 positives
{'min': 105.0, 'max': 105.0}


In [3]:
# byte-identical files with different labels can't both be right -> drop them
df["md5"] = [hashlib.md5(Path(p).read_bytes()).hexdigest() for p in df.path]
dups = df[df.md5.duplicated(keep=False)]
conflict = dups.groupby("md5").filter(lambda g: len(g[["v", "h"]].drop_duplicates()) > 1)
print(dups[["path", "v", "h"]].to_string() if len(dups) else "no duplicates")
df = df.drop(conflict.index).reset_index(drop=True)
print("after dropping conflicting duplicates:", len(df))

                                                                 path   v  h
633  ..\data\raw\Columbia Gaze Data Set\0007\0007_2m_-15P_-10V_0H.jpg -10  0
636  ..\data\raw\Columbia Gaze Data Set\0007\0007_2m_-15P_-10V_5H.jpg -10  5
after dropping conflicting duplicates: 5878


## 2. Geometry features (my own definitions)
Per eye, in pixel coordinates, with the eye axis running between the two corners (left-to-right in the image so both eyes share a sign):
- `along`: iris centre offset from the corner midpoint, along the eye axis, / eye width
- `across`: same offset perpendicular to the axis, / eye width
- `open`: upper-to-lower lid distance / eye width

Plus head yaw, pitch and roll from MediaPipe's facial transformation matrix. 9 features in total.
Images are decoded at half resolution (2592×1728) for speed.

In [4]:
EYES = {"r": (33, 133, 159, 145, 468), "l": (362, 263, 386, 374, 473)}  # corner, corner, upper lid, lower lid, iris centre

def features(lm, M, w, h):
    P = np.array([[p.x * w, p.y * h] for p in lm])
    out = {}
    for e, (a, b, up, lo, iris) in EYES.items():
        axis = P[b] - P[a]
        width = np.linalg.norm(axis)
        ax = axis / width
        d = P[iris] - (P[a] + P[b]) / 2
        out[f"{e}_along"] = d @ ax / width
        out[f"{e}_across"] = d @ np.array([-ax[1], ax[0]]) / width
        out[f"{e}_open"] = np.linalg.norm(P[up] - P[lo]) / width
    R = np.asarray(M)[:3, :3]
    out["yaw"] = np.degrees(np.arcsin(-R[2, 0]))
    out["pitch"] = np.degrees(np.arctan2(R[2, 1], R[2, 2]))
    out["roll"] = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    return out

if CACHE.exists():
    feats = pd.read_csv(CACHE, dtype={"subject": str})
else:
    opts = vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=str(MODEL)),
                                        num_faces=1, output_facial_transformation_matrixes=True)
    recs, t0 = [], time.time()
    with vision.FaceLandmarker.create_from_options(opts) as lmk:
        for i, p in enumerate(df.path):
            img = cv2.cvtColor(cv2.imread(p, cv2.IMREAD_REDUCED_COLOR_2), cv2.COLOR_BGR2RGB)
            res = lmk.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(img)))
            if res.face_landmarks:
                recs.append({"path": p, **features(res.face_landmarks[0], res.facial_transformation_matrixes[0], img.shape[1], img.shape[0])})
            if i % 1000 == 0:
                print(i, f"{time.time() - t0:.0f}s")
    feats = df.merge(pd.DataFrame(recs), on="path")
    feats.to_csv(CACHE, index=False)
FEATS = [c for c in feats.columns if c.startswith(("r_", "l_")) or c in ("yaw", "pitch", "roll")]
print(f"face found in {len(feats)}/{len(df)} images")
feats.groupby("y")[FEATS].mean().round(3)

0 0s


1000 61s


2000 122s


3000 183s


4000 244s


5000 305s


face found in 5878/5878 images


,r_along,r_across,r_open,l_along,l_across,l_open,yaw,pitch,roll
y,,,,,,,,,
0,-0.028,-0.074,0.345,0.045,-0.073,0.338,-0.964,-1.632,1.439
1,-0.026,-0.075,0.351,0.044,-0.074,0.342,-1.026,-0.867,1.363


## 3–5. Subject-disjoint CV, logistic regression, AUROC
GroupKFold(5) on subject ID. Logistic regression fixed in advance (C=1, balanced class weights), trained on all
training-fold images. AUROC is computed per fold (the repo reports the mean over folds) and on the pooled
out-of-fold scores with a 95% subject-bootstrap CI.

In [5]:
X, y, g = feats[FEATS].to_numpy(), feats.y.to_numpy(), feats.subject.to_numpy()
lr = lambda: LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000)
models = {"linear": lambda: make_pipeline(StandardScaler(), lr()),
          "poly2": lambda: make_pipeline(StandardScaler(), PolynomialFeatures(2), StandardScaler(), lr())}

rng = np.random.default_rng(0)
subjects = np.unique(g)
idx_by_subj = {s: np.flatnonzero(g == s) for s in subjects}
boot = [np.concatenate([idx_by_subj[s] for s in rng.choice(subjects, len(subjects))]) for _ in range(1000)]

summary, oof = [], {}
for name, make in models.items():
    score, fold_auc = np.zeros(len(y)), []
    for tr, te in GroupKFold(n_splits=5).split(X, y, g):
        assert not set(g[tr]) & set(g[te])
        m = make().fit(X[tr], y[tr])
        score[te] = m.predict_proba(X[te])[:, 1]
        fold_auc.append(roc_auc_score(y[te], score[te]))
    oof[name] = score
    ci = np.percentile([roc_auc_score(y[b], score[b]) for b in boot], [2.5, 97.5])
    summary.append(dict(model=name, mean_fold_auroc=np.mean(fold_auc), fold_sd=np.std(fold_auc),
                        pooled_auroc=roc_auc_score(y, score), ci_low=ci[0], ci_high=ci[1],
                        folds=" ".join(f"{a:.3f}" for a in fold_auc)))
pd.DataFrame(summary).round(3)

,model,mean_fold_auroc,fold_sd,pooled_auroc,ci_low,ci_high,folds
0,linear,0.551,0.024,0.546,0.530,0.564,0.554 0.578 0.515 0.575 0.536
1,poly2,0.812,0.011,0.808,0.786,0.831,0.798 0.827 0.799 0.817 0.818


In [6]:
# by head pose (paper: 0.891 frontal, 0.718-0.766 at ±30°)
feats.assign(score=oof["poly2"]).groupby("pose").apply(lambda d: roc_auc_score(d.y, d.score), include_groups=False).round(3).rename("poly2 AUROC").to_frame().T

pose,-30,-15,0,15,30
poly2 AUROC,0.714,0.844,0.887,0.833,0.77


In [7]:
# where does it fail? share predicted "camera" (p > 0.5) per gaze offset, poly2
pred = oof["poly2"] > 0.5
feats.assign(pred=pred).pivot_table(index="v", columns="h", values="pred", aggfunc="mean").round(2)

h,-15,-10,-5,0,5,10,15
v,,,,,,,
-10,0.09,0.24,0.49,0.59,0.43,0.18,0.07
0,0.12,0.31,0.65,0.79,0.61,0.26,0.05
10,0.11,0.25,0.52,0.65,0.51,0.19,0.08


## 6. What I did differently from the repo

Table written before running, from the repo's `docs/methodology.md`; result filled in after.

| | repo | this notebook |
|---|---|---|
| Folds | 56 subjects permuted with seed 20261108, dealt round-robin into 5 folds | `sklearn` `GroupKFold(5)` on subject ID, so the folds hold different people |
| MediaPipe input | landmarks also used to make 448² face crops for the other methods | Face Landmarker run directly on the full frame, decoded at half resolution (2592×1728) |
| Which images count | "commonly eligible": every one of the 4 methods had to produce features | every image where my own landmarker run found a face |
| Features | 9: iris along/across the eye axis and lid aperture per eye (÷ eye width), 3 head Euler angles | same 9 ideas, but my own landmark indices (33/133/159/145/468, 362/263/386/374/473) and my own Euler convention |
| Head | standardise → poly2 → standardise → LR(C=1, balanced) | same, plus the linear version |
| Training data | ALL = every training-fold image, averaged over 5 seeds (identical at ALL) | every training-fold image, one run |
| Headline number | mean of the per-fold AUROCs, fold-stratified subject bootstrap CI | mean of per-fold AUROCs (comparable), plus pooled out-of-fold AUROC with a plain subject bootstrap CI |
| Duplicate pair | excluded after the repo's audit | found with my own MD5 check; conflicting pair dropped |

Pooled AUROC ranks scores from 5 different models against each other, so it can be a bit below the fold mean.

**Result:** replicated. Repo numbers are from `results/tables/learning_curves.csv` and `head_pose_auroc.csv`.

| | repo | this notebook |
|---|---|---|
| poly2, mean fold AUROC (ALL) | 0.810 [0.791, 0.833] | **0.812** (fold SD 0.011); pooled 0.808 [0.786, 0.831] |
| linear, mean fold AUROC (ALL) | 0.555 [0.543, 0.574] | 0.551 (fold SD 0.024) |
| poly2 by head pose −30 / −15 / 0 / +15 / +30° | 0.718 / 0.835 / 0.891 / 0.847 / 0.766 | 0.714 / 0.844 / 0.887 / 0.833 / 0.770 |

The gap is 0.002, well inside the repo's CI and smaller than the spread between my own folds, so none of the differences
in the first table matter at this level. Different folds, half-resolution full frames instead of the repo's pipeline,
and my own landmark indices and Euler convention all give the same answer. MediaPipe found a face in all 5,878 images,
so eligibility made no difference either. My MD5 check found the same conflicting pair in subject 0007 that the repo's audit excluded.

The two things that matter are confirmed too. Without the polynomial the features are near chance (0.55), because the
positive class is a small region in the middle of the feature space and a single linear boundary can't enclose it.
The 5° boundary is still the weak point: with ALL labels the model says "camera" for 79 % of true positives but also for
61–65 % of images 5° to the side at the same height (last table above).